In [16]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.base import TaskResult
from autogen_core import CancellationToken
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GEMINI_KEY")

model_client = OpenAIChatCompletionClient(model="gemini-2.0-flash",api_key=api_key)


In [17]:

planning_agent = AssistantAgent(
    name = "planning_agent",
    model_client = model_client,
    description="You are a planning agent to create a plan for the task given by the user.",
    system_message= """
    You are a planning agent,
    Your job is to break down complex tasks into smaller sub tasks. 
    Your team members are:

            web_search_agent : Searches for the information in the web.
            data_analyst_agent: performs Calculations.
    You only plan and delegate task - You donot execute themselves.
    When assigning a task use the below format:
    1. <Agent_name>:<task>


    After all the tasks completed,summarize the findings and end with 'TERMINATE'

"""
)


In [18]:

def search_web_tool(query:str)-> str:
    """tools performs the  web search and returns the result.
        args:query 
        """
    if "2006-2007" in query:
        return """The 2006-2007 season was the 105th season of competitive football in England. Chelsea won the Premier League, while Liverpool won the FA Cup." \
        points to note:
        1. Chelsea won the Premier League.
        udonis points: 200"""
    elif "2007-2008" in query:
        return "The 2007-2008 season was the 106th season of competitive football in England. Manchester United won the Premier League, while Portsmouth won the FA Cup."
    else:
        return "No information available for the specified query."
   
WebSearchAgent = AssistantAgent(
    name = 'web_search_agent',
    description = "You are a great web search agent who solves the task using web.",
    model_client=model_client,
    system_message="""You are a web search agent,
    your only tool is to search the web and return the result.
    You are given a query and you need to search the web and return the result.
    Once you have the result, you never do calculations or analysis.""",
    tools=[search_web_tool],
)


In [19]:

def percentage_calculation_tool(start:float,end:float) -> float:
    """tools performs the percentage calculation and returns the result.
        args : start: float, end: float
        returns: float 
        """
    if start == 0:
        return 0.0
    return ((end - start) / start) * 100
    
DataAnalysisAgent = AssistantAgent(
    name = 'data_analyst_agent',
    description = "You are a great data analysis agent who solves the task.",
    model_client=model_client,
    system_message="""You are a data analysis agent,
    ,Given the data you have assigned.You should analyse the data 
    and provide the results using the tools provided.
    if you have not seen the data, you can ask for it.""",
    tools=[percentage_calculation_tool],
)


In [20]:
termination_condition = TextMentionTermination("TERMINATE")
from autogen_agentchat.conditions import MaxMessageTermination
max_message_termination = MaxMessageTermination(max_messages=20)
combined_termination = termination_condition | max_message_termination


Selector Prompt : Uses a model to select the next agent based on the assigned task.

In [21]:
custom_selector_prompt = """
Select an agent to perform the task.

{roles}

Conversation History:
{history}

Read the above Conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned task before other agents starts working.
Only Select one agent.

"""


Try not load overinformation with the selector prompt,

 participants: Names of the candidates. the format is ["\<name1>","\<name2>"]

 Roles: A list of names and description of the candidate agents.the format is "\<name>":"\<description>"

 History: the conversation history formmated as double new line seperated of names and message content.the format of each message is "\<name>:\<message_content>"

In [22]:
selector_team = SelectorGroupChat(
    participants=[planning_agent,WebSearchAgent,DataAnalysisAgent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=custom_selector_prompt,
    max_turns=5,
    allow_repeated_speaker=True
)



In [23]:
task = """who won during the 2006-2007 and 2007-2008 seasons of the English Premier League?"""



In [24]:
from autogen_agentchat.ui import Console
await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
who won during the 2006-2007 and 2007-2008 seasons of the English Premier League?
---------- ToolCallRequestEvent (web_search_agent) ----------
[FunctionCall(id='', arguments='{"query":"who won the English Premier League in 2006-2007 and 2007-2008 seasons?"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (web_search_agent) ----------
[FunctionExecutionResult(content='The 2006-2007 season was the 105th season of competitive football in England. Chelsea won the Premier League, while Liverpool won the FA Cup."         points to note:\n        1. Chelsea won the Premier League.\n        udonis points: 200', name='search_web_tool', call_id='', is_error=False)]
---------- ToolCallSummaryMessage (web_search_agent) ----------
The 2006-2007 season was the 105th season of competitive football in England. Chelsea won the Premier League, while Liverpool won the FA Cup."         points to note:
        1. Chelsea won the Premier League.
       

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 11, 23, 314510, tzinfo=datetime.timezone.utc), content='who won during the 2006-2007 and 2007-2008 seasons of the English Premier League?', type='TextMessage'), ToolCallRequestEvent(source='web_search_agent', models_usage=RequestUsage(prompt_tokens=117, completion_tokens=36), metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 11, 26, 235125, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='', arguments='{"query":"who won the English Premier League in 2006-2007 and 2007-2008 seasons?"}', name='search_web_tool')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='web_search_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 11, 26, 238125, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='The 2006-2007 season was the 105th season of competitive football in England. Chelsea won the

In [25]:
task = """find the percentage of 30 to 50 in the given data set"""

from autogen_agentchat.ui import Console
await Console(selector_team.run_stream(task=task))


---------- TextMessage (user) ----------
find the percentage of 30 to 50 in the given data set
---------- ToolCallRequestEvent (data_analyst_agent) ----------
[FunctionCall(id='', arguments='{"end":50,"start":30}', name='percentage_calculation_tool')]
---------- ToolCallExecutionEvent (data_analyst_agent) ----------
[FunctionExecutionResult(content='66.66666666666666', name='percentage_calculation_tool', call_id='', is_error=False)]
---------- ToolCallSummaryMessage (data_analyst_agent) ----------
66.66666666666666
---------- TextMessage (data_analyst_agent) ----------
The percentage of 30 to 50 is 66.67%.

---------- TextMessage (planning_agent) ----------
Okay, I need to calculate the percentage of 30 to 50.

Here's the plan:

1.  **data_analyst_agent**: Calculate the percentage of 30 to 50. Provide the result.

After the calculation is complete, I will provide a summary of the findings.

---------- TextMessage (planning_agent) ----------

---------- ToolCallRequestEvent (data_analys

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 12, 0, 380633, tzinfo=datetime.timezone.utc), content='find the percentage of 30 to 50 in the given data set', type='TextMessage'), ToolCallRequestEvent(source='data_analyst_agent', models_usage=RequestUsage(prompt_tokens=577, completion_tokens=9), metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 12, 3, 138243, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='', arguments='{"end":50,"start":30}', name='percentage_calculation_tool')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='data_analyst_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 12, 3, 140240, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='66.66666666666666', name='percentage_calculation_tool', call_id='', is_error=False)], type='ToolCallExecutionEvent'), ToolCallSummaryMessage(source='data_analyst_agent', 

For having the better control over the custom selector. we use the custom_selector_func 

we are going to call planning agent after specialized agent works.


In [26]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
def selector_function(messages: Sequence[BaseAgentEvent|BaseChatMessage])->str|None:
    if messages[-1].source !=planning_agent.name:
        return planning_agent.name
    return None



In [27]:
await selector_team.reset()


In [28]:
selector_team = SelectorGroupChat(
    participants=[planning_agent,WebSearchAgent,DataAnalysisAgent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=custom_selector_prompt,
    max_turns=5,
    allow_repeated_speaker=True,
    selector_func=selector_function
)


In [31]:
task = """find the percentage of 30 to 50 and provide me the result"""

from autogen_agentchat.ui import Console
await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
find the percentage of 30 to 50 and provide me the result
---------- TextMessage (planning_agent) ----------
Okay, I understand the request. Here's the plan to find the percentage of 30 to 50:

1. data_analyst_agent: Calculate the percentage of 30 to 50 and provide the result.
2. Summarize the findings.

TERMINATE



TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 13, 17, 832995, tzinfo=datetime.timezone.utc), content='find the percentage of 30 to 50 and provide me the result', type='TextMessage'), TextMessage(source='planning_agent', models_usage=RequestUsage(prompt_tokens=287, completion_tokens=62), metadata={}, created_at=datetime.datetime(2025, 7, 29, 10, 13, 18, 970918, tzinfo=datetime.timezone.utc), content="Okay, I understand the request. Here's the plan to find the percentage of 30 to 50:\n\n1. data_analyst_agent: Calculate the percentage of 30 to 50 and provide the result.\n2. Summarize the findings.\n\nTERMINATE\n", type='TextMessage')], stop_reason="Text 'TERMINATE' mentioned")